In [ ]:
# Import libraries

import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
import math
from scipy.special import comb

In [ ]:
# Set parameters

r = 0.05  # Risk-free interest rate
sigma = 0.2  # Stock price volatility
Time = 1  # Time to expiry (in years)
k = 105 # Strike price
S0 = 100  # Initial stock price
n_replications = 1000000  # Number of simulation replications
n_nodes = 1000 # Number of steps for binomial method

# European Stock Option

Monte Carlo Method

In [ ]:
def monte_carlo_estimate_option_value(r, sigma, Time, k, S0, n_replications, option_type = 'call'):
    np.random.seed(1234)
    Z = np.random.standard_normal(n_replications)
    drift = (r - 0.5 * sigma**2) * Time
    diffusion = sigma * np.sqrt(Time) * Z
    ST = S0 * np.exp(drift + diffusion)
    if option_type == 'call':
      payoff = np.maximum(ST - k, 0)
    elif option_type == 'put':
      payoff = np.maximum(k - ST, 0)
    option_value = np.exp(-r * Time) * np.mean(payoff)
    return option_value


option_value_european = monte_carlo_estimate_option_value(r, sigma, Time, k, S0, n_replications)
print("Estimated European option value - Monte Carlo:", option_value_european)

Estimated European option value - Monte Carlo: 8.023765933620492


Log Normal Method

In [ ]:
def lognormal_option_price_european(r, sigma, Time, k, S0, n_replications, option_type = 'call'):
    np.random.seed(1234)
    drift = (r - 0.5 * sigma**2) * Time
    diffusion_scaling = sigma * np.sqrt(Time)
    lognormal_samples = np.random.lognormal(mean=drift, sigma=diffusion_scaling, size=n_replications)
    ST = S0 * lognormal_samples
    if option_type == 'call':
      payoff = np.maximum(ST - k, 0)
    elif option_type == 'put':
      payoff = np.maximum(k - ST, 0)
    option_value = np.exp(-r * Time) * np.mean(payoff)
    return option_value

option_value_lognormal_european = lognormal_option_price_european(r, sigma, Time, k, S0, n_replications)
print("Estimated European option value - Lognormal:", option_value_lognormal_european)

Estimated European option value - Lognormal: 8.023765933620492


Binomial Method

In [ ]:
def binomial_option_value_european(S0, k, Time, r, sigma, n_nodes, type_ = 'call'):
    dt = Time/n_nodes
    u = np.exp(sigma * np.sqrt(dt))
    d = np.exp(-sigma * np.sqrt(dt))
    p = (  np.exp(r*dt) - d )  /  (  u - d )
    payoff = 0
    for i in range(n_nodes + 1):
        node_prob = comb(n_nodes, i, exact = True)*(p**i)*((1 - p)**(n_nodes - i))
        ST = S0*(u)**i*(d)**(n_nodes - i)
        if type_ == 'call':
            payoff += max(ST - k,0) * node_prob
        elif type_ == 'put':
            payoff += max(k - ST, 0) * node_prob
    option_value = np.exp(-r * Time) * payoff
    return option_value


option_value_binomial_european = binomial_option_value_european(S0, k, Time, r, sigma, n_nodes)
print("Estimated European option value - Binomial", option_value_binomial_european)

Estimated European option value - Binomial 8.021060287697196


Black-Scholes Method

In [ ]:
def black_scholes_option_price_european(r, sigma, Time, k, S0, option_type = 'call'):
  b = (r*Time - (sigma**2 * Time) / 2 - np.log(k / S0)) / (sigma * np.sqrt(Time))
  if option_type == 'call':
    black_scholes = S0 * norm.cdf(b + sigma*np.sqrt(Time)) - k * np.exp(-r * Time) * norm.cdf(b)
  elif option_type == 'put':
    black_scholes = k * np.exp(-r * Time) * norm.cdf(b) - S0 * norm.cdf(b + sigma*np.sqrt(Time))
  return black_scholes

option_value_black_scholes_european = black_scholes_option_price_european(r, sigma, Time, k, S0)
print("Black-Scholes option value:", option_value_black_scholes_european)

Black-Scholes option value: 8.021352235143176


#Option Greeks

In [ ]:
d1 = (np.log(S0 / k) + (r + 0.5 * sigma ** 2) * Time) / (sigma * np.sqrt(Time))
d2 = d1 - sigma * np.sqrt(Time)

def calculate_delta(S0, k, r, Time, sigma, option_type = 'call'):
  if option_type == 'call':
       delta = norm.cdf(d1)
  elif option_type == 'put':
      delta = norm.cdf(d1) - 1
  return delta

def calculate_gamma(S0, k, r, Time, sigma):
    gamma = norm.pdf(d1) / (S0 * sigma * np.sqrt(Time))
    return gamma

def calculate_theta(S0, k, r, Time, sigma, option_type = 'call'):
    if option_type == 'call':
        theta = -(S0 * norm.pdf(d1) * sigma / (2 * np.sqrt(Time))) - r * k * np.exp(-r * Time) * norm.cdf(d2)
    elif option_type == 'put':
        theta = -(S0 * norm.pdf(d1) * sigma / (2 * np.sqrt(Time))) + r * k * np.exp(-r * Time) * norm.cdf(-d2)
    return theta / 365  # Convert to daily theta

def calculate_vega(S0, k, r, Time, sigma):
    vega = S0 * norm.pdf(d1) * np.sqrt(Time)
    return vega / 100  # Convert to vega per percentage point change in volatility

def calculate_rho(S0, k, r, Time, sigma, option_type = 'call'):
    if option_type == 'call':
        rho = Time * k * np.exp(-r * Time) * norm.cdf(d2)
    elif option_type == 'put':
        rho = -Time * k * np.exp(-r * Time) * norm.cdf(-d2)
    return rho / 100  # Convert to rho per percentage point change in interest rate


# Calculate Greeks for a call option
call_delta = calculate_delta(S0, k, r, Time, sigma)
call_gamma = calculate_gamma(S0, k, r, Time, sigma)
call_theta = calculate_theta(S0, k, r, Time, sigma)
call_vega = calculate_vega(S0, k, r, Time, sigma)
call_rho = calculate_rho(S0, k, r, Time, sigma)


# Print results
print("Call Option Greeks:")
print(f"Delta: {call_delta:.4f}")
print(f"Gamma: {call_gamma:.4f}")
print(f"Theta: {call_theta:.4f}")
print(f"Vega: {call_vega:.4f}")
print(f"Rho: {call_rho:.4f}")

Call Option Greeks:
Delta: 0.5422
Gamma: 0.0198
Theta: -0.0172
Vega: 0.3967
Rho: 0.4620


In [ ]:
# Calculate Greeks for a put option
option_type = 'put'
put_delta = calculate_delta(S0, k, r, Time, sigma, option_type)
put_gamma = calculate_gamma(S0, k, r, Time, sigma)
put_theta = calculate_theta(S0, k, r, Time, sigma, option_type)
put_vega = calculate_vega(S0, k, r, Time, sigma)
put_rho = calculate_rho(S0, k, r, Time, sigma, option_type)

print("\nPut Option Greeks:")
print(f"Delta: {put_delta:.4f}")
print(f"Gamma: {put_gamma:.4f}")
print(f"Theta: {put_theta:.4f}")
print(f"Vega: {put_vega:.4f}")
print(f"Rho: {put_rho:.4f}")


Put Option Greeks:
Delta: -0.4578
Gamma: 0.0198
Theta: -0.0035
Vega: 0.3967
Rho: -0.5368
